In [640]:
import pandas as pd
import geopandas as gpd
from shapely import wkt
import numpy as np

In [641]:
bus = pd.read_csv('Nodes/Bus.csv')
fgc = pd.read_csv('Nodes/FGC.csv')
metro = pd.read_csv('Nodes/Metro.csv')
tram = pd.read_csv('Nodes/Tram.csv')

all_stops = pd.concat([ bus, fgc, metro, tram], ignore_index=True)

exchange_edges = pd.read_csv('Edges/Exchanges.csv')

In [642]:
exchange_edges = exchange_edges.merge(all_stops[['id','stop_id']], left_on='dest', right_on='id', how='left')

# Tram

In [643]:
exchange_edges_tram = exchange_edges[exchange_edges['dest'].str.startswith('T')]

In [644]:
stop_times_x = pd.read_table('Data/Tram/TBX/stop_times.txt', sep=',')
stop_times_S = pd.read_table('Data/Tram/TBS/stop_times.txt', sep=',')
stop_times = pd.concat([stop_times_x, stop_times_S])    
stop_times = stop_times[stop_times['departure_time'].notna()]
stop_times = stop_times[['trip_id','stop_id','departure_time']]
stop_times = stop_times[stop_times['departure_time'].str.slice(0,2).astype(int) >= 7]
stop_times = stop_times[stop_times['departure_time'].str.slice(0,2).astype(int) < 11]

In [645]:
stop_times

,trip_id,stop_id,departure_time
71,2555_0196,1127,07:01:00
72,2555_0001,1127,07:06:30
73,2555_0001,1126,07:08:10
74,2555_0001,1125,07:10:10
75,2555_0001,1122,07:12:10
...,...,...,...
10511,1910_0451,2017,10:47:50
10512,1910_0451,2019,10:49:40
10513,1910_0452,2019,10:55:00
10514,1910_0452,2117,10:57:10


In [646]:
stops_x = pd.read_table('Data/Tram/TBX/stops.txt', sep=',')[['stop_id','stop_name','stop_desc']]
stops_s = pd.read_table('Data/Tram/TBS/stops.txt', sep=',')[['stop_id','stop_name','stop_desc']]
stops = pd.concat([stops_x, stops_s])
stops = stops[~stops['stop_id'].str.startswith('S')]
stops['stop_id'] = stops['stop_id'].astype(int)

In [647]:
stops_w_times = stops.merge(stop_times, on='stop_id', how='left')
stops_w_times = stops_w_times[stops_w_times['trip_id'].notna()]

In [648]:
trips_x = pd.read_table('Data/Tram/TBX/trips.txt', sep=',')
trips_s = pd.read_table('Data/Tram/TBS/trips.txt', sep=',')
trips = pd.concat([trips_x, trips_s])
trips = trips[['route_id','trip_id']]

In [649]:
times_w_routes = stops_w_times.merge(trips, on='trip_id', how='left')
times_w_routes.drop_duplicates(subset=['stop_id','stop_name','route_id','departure_time'], keep='first', inplace=True)
times_w_routes['route_id'] = 'T' + times_w_routes['route_id'].astype(int).astype(str)
times_w_routes['stop_desc'] = times_w_routes['stop_desc'].str[2:]
times_w_routes['stop_desc'] = times_w_routes['stop_desc'].replace({'RIGL':'SRMN','LLEV':'BDOR','MRSM':'DGMR','CATA':'CTLN','JOAN':'STJB','MRTI':'STMR','SROC':'STRC'})
times_w_routes['departure_time'] = pd.to_datetime(times_w_routes['departure_time'], format='%H:%M:%S')
times_w_routes = times_w_routes.sort_values(['stop_id', 'route_id', 'departure_time'])

times_w_routes['interval_minutes'] = (
    times_w_routes.groupby(['stop_id', 'route_id'])['departure_time']
    .diff()
    .dt.total_seconds()
    .div(60)
)
times_w_routes['interval_minutes'] = times_w_routes['interval_minutes'].apply(lambda x: np.nan if x < 0 else x)
avg_int_len = (
    times_w_routes.dropna(subset=['interval_minutes'])
    .groupby(['route_id', 'stop_name', 'stop_desc'], as_index=False)['interval_minutes']
    .mean()
).rename(columns={'interval_minutes': 'avg_interval_minutes'})
int_var = times_w_routes.groupby([ 'route_id', 'stop_name', 'stop_desc'], as_index=False)['interval_minutes'].var().rename(columns={'interval_minutes': 'var_interval_minutes'})
avg_wait_times = avg_int_len.merge(int_var, on=['route_id', 'stop_name', 'stop_desc'], how='left')
avg_wait_times['wait_time'] =((avg_wait_times['avg_interval_minutes'] ** 2) + avg_wait_times['var_interval_minutes']) /( 2*avg_wait_times['avg_interval_minutes'])
avg_wait_times['id'] = 'T' + '-' + avg_wait_times['route_id'] + '-' + avg_wait_times['stop_desc']
avg_wait_times = avg_wait_times[['id', 'wait_time']]
avg_wait_times

,id,wait_time
0,T-T1-XILE,5.534852
1,T-T1-BVIA,3.198554
2,T-T1-OLIV,5.663828
3,T-T1-CLOT,5.663828
4,T-T1-SRMN,5.569308
...,...,...
81,T-T6-SABS,4.420414
82,T-T6-MINA,8.245186
83,T-T6-PARC,8.245186
84,T-T6-PORT,8.245186


In [650]:
exchange_edges_tram = exchange_edges_tram.merge(avg_wait_times, left_on='dest', right_on='id', how='left')
exchange_edges_tram = exchange_edges_tram[['origen', 'dest', 'tram', 'mode','lines','type','time',
                                           'wait_time','directed','geometry']]
exchange_edges_tram

,origen,dest,tram,mode,lines,type,time,wait_time,directed,geometry
0,ST-FRMC,T-T1-FRMC,Francesc Macià - Francesc Macià,Tram - Tram,IU Stop - T1,Exchange - Self,2.000000,6.601392,True,POINT (2.143174886703491 41.3922004699707)
1,T-T2-FRMC,T-T1-FRMC,Francesc Macià - Francesc Macià,Tram - Tram,T2 - T1,Exchange - Self,1.000000,6.601392,True,POINT (2.143174886703491 41.3922004699707)
2,T-T3-FRMC,T-T1-FRMC,Francesc Macià - Francesc Macià,Tram - Tram,T3 - T1,Exchange - Self,1.000000,6.601392,True,POINT (2.143174886703491 41.3922004699707)
3,ST-FRMC,T-T2-FRMC,Francesc Macià - Francesc Macià,Tram - Tram,IU Stop - T2,Exchange - Self,2.000000,4.218604,True,POINT (2.143174886703491 41.3922004699707)
4,T-T1-FRMC,T-T2-FRMC,Francesc Macià - Francesc Macià,Tram - Tram,T1 - T2,Exchange - Self,1.000000,4.218604,True,POINT (2.143174886703491 41.3922004699707)
...,...,...,...,...,...,...,...,...,...,...
1888,B-V33-3596,T-T6-MINA,La Mina - Ponent - La Mina,Bus - Tram,V33 - T6,Exchange,4.566667,8.245186,True,"LINESTRING (2.2172843 41.4188065, 2.2174015 41..."
1889,B-B23-105999,T-T6-MINA,Rambla de la Mina - La Mina,Bus - Tram,B23 - T6,Exchange,1.800000,8.245186,True,"LINESTRING (2.2224278 41.4182191, 2.222341 41...."
1890,B-B23-106000,T-T6-MINA,CAP La Mina - La Mina,Bus - Tram,B23 - T6,Exchange,4.166667,8.245186,True,"LINESTRING (2.2201838 41.4156256, 2.2204138 41..."
1891,T-T5-PARC,T-T6-MINA,Parc del Besòs - La Mina,Tram - Tram,T5 - T6,Exchange,4.850000,8.245186,True,"LINESTRING (2.2171297 41.4191608, 2.2171685 41..."


# FCG

In [651]:
exchange_edges_fgc = exchange_edges[exchange_edges['dest'].str.startswith('F')]

In [652]:
stop_times = pd.read_table('Data/FGC/gtfs_fgc/stop_times.txt', sep=',')
stop_times = stop_times[['trip_id','stop_id','departure_time']]
stop_times = stop_times[stop_times['departure_time'].str.slice(0,2).astype(int) >= 7]
stop_times = stop_times[stop_times['departure_time'].str.slice(0,2).astype(int) < 11]

In [653]:
stops = pd.read_table('Data/FGC/gtfs_fgc/stops.txt', sep=',')[['stop_id','stop_name']]

In [654]:
stops_w_times = stops.merge(stop_times, on='stop_id', how='left')
stops_w_times = stops_w_times[stops_w_times['trip_id'].notna()]

In [655]:
trips = pd.read_table('Data/FGC/gtfs_fgc/trips.txt', sep=',')[['route_id','trip_id','trip_headsign']]
trips = trips[['route_id','trip_id','trip_headsign']]
valid = ['L6', 'L7', 'L8', 'L12','S1']
trips = trips[trips['route_id'].isin(valid)]

In [656]:
times_w_routes = trips.merge(stops_w_times, on='trip_id', how='left')
times_w_routes.drop_duplicates(subset=['route_id','stop_name','departure_time','trip_headsign'], keep='first', inplace=True)
times_w_routes = times_w_routes[times_w_routes['departure_time'].notna()]
times_w_routes['stop_id']  = times_w_routes['stop_id'].str[:2]
times_w_routes

,route_id,trip_id,trip_headsign,stop_id,stop_name,departure_time
8,L6,6c4bdae602747613ef|6f2dc7e30b,Sarrià,MN,Muntaner,07:00:00
9,L6,6c4bdae602747613ef|6f2dc7e30b,Sarrià,BN,La Bonanova,07:01:00
10,L6,6c4bdae602747613ef|6f2dc7e30b,Sarrià,TT,Les Tres Torres,07:03:00
11,L6,6c4bdae602747613ef|6f2dc7e30b,Sarrià,SR,Sarrià,07:04:00
13,L6,6c4bdae602747613ef|6f2dc7e203,Sarrià,PC,Barcelona - Plaça Catalunya,07:07:00
...,...,...,...,...,...,...
29661,L8,625cdae11f726b1abb56|622dc5e102,Barcelona - Plaça Espanya,AL,Almeda,08:47:15
29662,L8,625cdae11f726b1abb56|622dc5e102,Barcelona - Plaça Espanya,CO,Cornellà Riera,08:45:15
29663,L8,625cdae11f726b1abb56|622dc5e102,Barcelona - Plaça Espanya,BO,Sant Boi,08:42:00
29664,L8,625cdae11f726b1abb56|622dc5e102,Barcelona - Plaça Espanya,EU,Europa | Fira,08:55:00


In [657]:
times_w_routes['departure_time'] = pd.to_datetime(times_w_routes['departure_time'], format='%H:%M:%S')
times_w_routes = times_w_routes.sort_values(['stop_id', 'route_id','trip_headsign', 'departure_time'])

times_w_routes['interval_minutes'] = (
    times_w_routes.groupby(['stop_id', 'route_id','trip_headsign'])['departure_time']
    .diff()
    .dt.total_seconds()
    .div(60)
)
 

times_w_routes['interval_minutes'] = times_w_routes['interval_minutes'].apply(lambda x: np.nan if x < 0 else x)
avg_int_len = (
    times_w_routes.dropna(subset=['interval_minutes'])
    .groupby(['route_id', 'stop_id', 'stop_name'], as_index=False)['interval_minutes']
    .mean()
).rename(columns={'interval_minutes': 'avg_interval_minutes'})
int_var = times_w_routes.groupby([ 'route_id', 'stop_id', 'stop_name'], as_index=False)['interval_minutes'].var().rename(columns={'interval_minutes': 'var_interval_minutes'})
avg_wait_times = avg_int_len.merge(int_var, on=['route_id', 'stop_id', 'stop_name'], how='left')
avg_wait_times['wait_time'] =((avg_wait_times['avg_interval_minutes'] ** 2) + avg_wait_times['var_interval_minutes']) /( 2*avg_wait_times['avg_interval_minutes'])



avg_wait_times = avg_wait_times.groupby(['route_id', 'stop_id', 'stop_name'], as_index=False)['wait_time'].mean()
avg_wait_times['id'] = 'F' + '-' + avg_wait_times['route_id'] + '-' + avg_wait_times['stop_id']
avg_wait_times

,route_id,stop_id,stop_name,wait_time,id
0,L12,RE,Reina Elisenda,2.042273,F-L12-RE
1,L12,SR,Sarrià,2.042273,F-L12-SR
2,L6,BN,La Bonanova,2.368944,F-L6-BN
3,L6,GR,Gràcia,1.924447,F-L6-GR
4,L6,MN,Muntaner,2.367738,F-L6-MN
5,L6,PC,Barcelona - Plaça Catalunya,1.923921,F-L6-PC
6,L6,PR,Provença,1.908047,F-L6-PR
7,L6,SG,Sant Gervasi,2.369616,F-L6-SG
8,L6,SR,Sarrià,2.380794,F-L6-SR
9,L6,TT,Les Tres Torres,2.377072,F-L6-TT


In [658]:
exchange_edges_fgc = exchange_edges_fgc.merge(avg_wait_times[['id','wait_time']], left_on='dest', right_on='id', how='left')
exchange_edges_fgc = exchange_edges_fgc[['origen', 'dest', 'tram', 'mode','lines','type','time',
                                           'wait_time','directed','geometry']]

# Bus

In [659]:
exchange_edges_bus = exchange_edges[exchange_edges['dest'].str.startswith('B')]
exchange_edges_bus

,origen,dest,tram,mode,lines,type,time,directed,geometry,id,stop_id
0,SB-2,B-59-2,Av Icària - Àlaba - Av Icària - Àlaba,Bus - Bus,IU Stop - 59,Exchange - Self,1.000000,True,POINT (2.198984998801282 41.393104003754864),B-59-2,2
1,B-H16-2,B-59-2,Av Icària - Àlaba - Av Icària - Àlaba,Bus - Bus,H16 - 59,Exchange - Self,1.000000,True,POINT (2.198984998801282 41.393104003754864),B-59-2,2
2,B-V27-2,B-59-2,Av Icària - Àlaba - Av Icària - Àlaba,Bus - Bus,V27 - 59,Exchange - Self,1.000000,True,POINT (2.198984998801282 41.393104003754864),B-59-2,2
3,SB-2,B-H16-2,Av Icària - Àlaba - Av Icària - Àlaba,Bus - Bus,IU Stop - H16,Exchange - Self,1.000000,True,POINT (2.198984998801282 41.393104003754864),B-H16-2,2
4,B-59-2,B-H16-2,Av Icària - Àlaba - Av Icària - Àlaba,Bus - Bus,59 - H16,Exchange - Self,1.000000,True,POINT (2.198984998801282 41.393104003754864),B-H16-2,2
...,...,...,...,...,...,...,...,...,...,...,...
147710,B-B15-108088,B-B81-112736,Raval - Santa Rosa - Elcano - Roger de Llúria,Bus - Bus,B15 - B81,Exchange,4.400000,True,"LINESTRING (2.2135373 41.4439583, 2.2134839 41...",B-B81-112736,112736
147711,B-B81-109790,B-B81-112736,Escola Pere de Tera - Elcano - Roger de Llúria,Bus - Bus,B81 - B81,Exchange,3.650000,True,"LINESTRING (2.214254 41.4408375, 2.2142405 41....",B-B81-112736,112736
147712,B-B81-109968,B-B81-112736,CAP Santa Rosa - Elcano - Roger de Llúria,Bus - Bus,B81 - B81,Exchange,1.800000,True,"LINESTRING (2.214348 41.4412638, 2.2142369 41....",B-B81-112736,112736
147713,B-B15-110507,B-B81-112736,Raval - Centre Cívic - Elcano - Roger de Llúria,Bus - Bus,B15 - B81,Exchange,3.616667,True,"LINESTRING (2.2108499 41.44294, 2.2109213 41.4...",B-B81-112736,112736


## AMB

In [660]:
stop_times = pd.read_table('Data/GTFS_AMB/stop_times.txt', sep=',')
stop_times = stop_times[['trip_id', 'stop_id', 'departure_time']].dropna()

stop_times['departure_time'] = pd.to_datetime(
    stop_times['departure_time'],
    format='%H:%M:%S',
    errors='coerce'
)

stop_times = stop_times[stop_times['departure_time'].dt.hour.between(7, 11)]
stop_times

,trip_id,stop_id,departure_time
0,129.20.1.1.0,107229,1900-01-01 08:55:00
1,129.20.1.1.0,2234,1900-01-01 08:56:09
2,129.20.1.1.0,2235,1900-01-01 08:57:46
3,129.20.1.1.0,476,1900-01-01 08:58:45
4,129.20.1.1.0,365,1900-01-01 08:59:57
...,...,...,...
890865,516.8.2.4.23,109921,1900-01-01 11:53:39
890866,516.8.2.4.23,106867,1900-01-01 11:54:41
890867,516.8.2.4.23,106868,1900-01-01 11:55:16
890868,516.8.2.4.23,106869,1900-01-01 11:57:03


In [661]:
stops = pd.read_table('Data/GTFS_AMB/stops.txt', sep=',')[['stop_id','stop_name']]
stops

,stop_id,stop_name
0,109303,Eusebi Güell - Joaquim Auger
1,100005,Escola Busquets i Punset
2,109239,Faigs - Montseny
3,109244,Faigs - Freixe
4,1461,Av de Cornellà - Pont d'Esplugues
...,...,...
4922,100031,Palau Reial
4923,100032,Zona Universitària
4924,112179,Pl. Jacint Verdaguer
4925,112099,Dr. Robert - Av. Bufalà


In [662]:
stops_w_times = stops.merge(stop_times, on='stop_id', how='left')
stops_w_times = stops_w_times[stops_w_times['trip_id'].notna()]
stops_w_times

,stop_id,stop_name,trip_id,departure_time
0,109303,Eusebi Güell - Joaquim Auger,196.25.2.1.0,1900-01-01 09:17:44
1,109303,Eusebi Güell - Joaquim Auger,196.25.2.1.1,1900-01-01 09:57:44
2,109303,Eusebi Güell - Joaquim Auger,196.25.2.1.2,1900-01-01 10:37:44
3,109303,Eusebi Güell - Joaquim Auger,196.25.2.1.3,1900-01-01 11:17:44
4,109303,Eusebi Güell - Joaquim Auger,196.25.2.1.4,1900-01-01 11:57:44
...,...,...,...,...
246520,112100,Dr. Robert - Sardana,297.13.1.1.14,1900-01-01 10:55:48
246521,112100,Dr. Robert - Sardana,297.13.1.1.15,1900-01-01 11:10:48
246522,112100,Dr. Robert - Sardana,297.13.1.1.16,1900-01-01 11:25:48
246523,112100,Dr. Robert - Sardana,297.13.1.1.17,1900-01-01 11:40:48


In [663]:
trips = pd.read_table('Data/GTFS_AMB/trips.txt', sep=',')
trips = trips[['route_id','trip_id','trip_headsign']]
routes = pd.read_table('Data/GTFS_AMB/routes.txt', sep=',')[['route_id','route_short_name']] 
trips = trips.merge(routes, on='route_id', how='left')

In [664]:
stops_w_times

,stop_id,stop_name,trip_id,departure_time
0,109303,Eusebi Güell - Joaquim Auger,196.25.2.1.0,1900-01-01 09:17:44
1,109303,Eusebi Güell - Joaquim Auger,196.25.2.1.1,1900-01-01 09:57:44
2,109303,Eusebi Güell - Joaquim Auger,196.25.2.1.2,1900-01-01 10:37:44
3,109303,Eusebi Güell - Joaquim Auger,196.25.2.1.3,1900-01-01 11:17:44
4,109303,Eusebi Güell - Joaquim Auger,196.25.2.1.4,1900-01-01 11:57:44
...,...,...,...,...
246520,112100,Dr. Robert - Sardana,297.13.1.1.14,1900-01-01 10:55:48
246521,112100,Dr. Robert - Sardana,297.13.1.1.15,1900-01-01 11:10:48
246522,112100,Dr. Robert - Sardana,297.13.1.1.16,1900-01-01 11:25:48
246523,112100,Dr. Robert - Sardana,297.13.1.1.17,1900-01-01 11:40:48


In [665]:
times_w_routes_amb = stops_w_times.merge(trips, on='trip_id', how='left')
times_w_routes_amb = times_w_routes_amb[times_w_routes_amb['departure_time'].notna()]
times_w_routes_amb.drop_duplicates(subset=['route_id','stop_name','departure_time'], keep='first', inplace=True)
times_w_routes_amb['stop_id']  =  times_w_routes_amb['stop_id'].astype(int).astype(str)
times_w_routes_amb

,stop_id,stop_name,trip_id,departure_time,route_id,trip_headsign,route_short_name
0,109303,Eusebi Güell - Joaquim Auger,196.25.2.1.0,1900-01-01 09:17:44,196,St. Boi L.,L74
1,109303,Eusebi Güell - Joaquim Auger,196.25.2.1.1,1900-01-01 09:57:44,196,St. Boi L.,L74
2,109303,Eusebi Güell - Joaquim Auger,196.25.2.1.2,1900-01-01 10:37:44,196,St. Boi L.,L74
3,109303,Eusebi Güell - Joaquim Auger,196.25.2.1.3,1900-01-01 11:17:44,196,St. Boi L.,L74
4,109303,Eusebi Güell - Joaquim Auger,196.25.2.1.4,1900-01-01 11:57:44,196,St. Boi L.,L74
...,...,...,...,...,...,...,...
244405,112100,Dr. Robert - Sardana,297.13.1.1.14,1900-01-01 10:55:48,297,Bonavista,B8
244406,112100,Dr. Robert - Sardana,297.13.1.1.15,1900-01-01 11:10:48,297,Bonavista,B8
244407,112100,Dr. Robert - Sardana,297.13.1.1.16,1900-01-01 11:25:48,297,Bonavista,B8
244408,112100,Dr. Robert - Sardana,297.13.1.1.17,1900-01-01 11:40:48,297,Bonavista,B8


## TMB

In [666]:
stop_times = pd.read_table('Data/GTFS_TMB/stop_times.txt', sep=',')[['trip_id', 'stop_id', 'departure_time']]
stop_times = stop_times[stop_times['departure_time'].notna()]
stop_times = stop_times[stop_times['departure_time'].str.slice(0,2).astype(int) >= 7]
stop_times = stop_times[stop_times['departure_time'].str.slice(0,2).astype(int) <= 11]
stop_times['stop_id'] = stop_times['stop_id'].astype(str)
stop_times = stop_times[stop_times['trip_id'].str.startswith('2')]
stop_times

/var/folders/95/s4thp5290fd41pw2cj8713k80000gn/T/ipykernel_51408/3576717721.py:1: DtypeWarning: Columns (3) have mixed types. Specify dtype option on import or set low_memory=False.
  stop_times = pd.read_table('Data/GTFS_TMB/stop_times.txt', sep=',')[['trip_id', 'stop_id', 'departure_time']]


,trip_id,stop_id,departure_time
777221,2.1.121.3345302.4094,2.9678.700773,10:05:00
777222,2.1.121.3345302.4094,2.9680.700539,10:10:00
777223,2.1.121.3345302.4094,2.8319.700682,10:19:00
777224,2.1.121.3345302.4094,2.1846.700880,10:21:00
777225,2.1.121.3345302.4094,2.9937.700777,10:24:00
...,...,...,...
1827645,2.250.102.3185372.3051,2.1595.694767,11:08:00
1827646,2.250.102.3185373.3051,2.1595.694767,11:18:00
1827651,2.250.102.3185373.3051,2.1622.691609,11:26:00
1827660,2.250.102.3185373.3051,2.819.695633,11:39:00


In [667]:
stops = pd.read_table('Data/GTFS_TMB/stops.txt', sep=',')[['stop_id','stop_name']]
stops['stop_id'] = stops['stop_id'].astype(str)

In [668]:
stops_w_times = stop_times.merge(stops, on='stop_id', how='left')

In [669]:
trips = pd.read_table('Data/GTFS_TMB/trips.txt', sep=',')[['route_id','trip_id','trip_headsign']]
routes = pd.read_table('Data/GTFS_TMB/routes.txt', sep=',')[['route_id','route_short_name']]
metro = ['L1','L2','L3','L4','L5','L9','L10','L11','L9N','L9S','L10N','L10S','FM','M1','M9','978']
trips = routes.merge(trips, on='route_id', how='left')
trips = trips[~trips['route_short_name'].isin(metro)]
trips

,route_id,route_short_name,trip_id,trip_headsign
40199,2.220.2999,D20,2.220.64.3369726.2999,Ernest Lluch
40200,2.220.2999,D20,2.220.64.3369728.2999,Ernest Lluch
40201,2.220.2999,D20,2.220.64.3369724.2999,Ernest Lluch
40202,2.220.2999,D20,2.220.64.3369708.2999,Ernest Lluch
40203,2.220.2999,D20,2.220.64.3369678.2999,Ernest Lluch
...,...,...,...,...
80906,2.196.2971,196,2.196.26.3064023.2971,Av. Tibidabo
80907,2.196.2971,196,2.196.26.3063966.2971,Av. Tibidabo
80908,2.196.2971,196,2.196.26.3064008.2971,Av. Tibidabo
80909,2.196.2971,196,2.196.26.3063968.2971,Av. Tibidabo


short routes, routes that don't represent real lines (ex trip Id = 2.239.105.3380438.4092)

In [670]:
times_w_routes_tmb = stops_w_times.merge(trips, on='trip_id', how='left')
times_w_routes_tmb = times_w_routes_tmb[times_w_routes_tmb['route_id'].notna()]
times_w_routes_tmb['stop_id'] = times_w_routes_tmb['stop_id'].str.split('.').str[1]
times_w_routes_tmb

,trip_id,stop_id,departure_time,stop_name,route_id,route_short_name,trip_headsign
1404,2.6.57.3060348.2899,3257,09:00:00,Llacuna - Ramon Turró - Final de trajecte,2.6.2899,6,Pg. Manuel Girona
1405,2.6.57.3060348.2899,3400,09:07:00,Marina - Meridiana,2.6.2899,6,Pg. Manuel Girona
1406,2.6.57.3060348.2899,317,09:15:00,Pl Tetuan - Pg de Sant Joan,2.6.2899,6,Pg. Manuel Girona
1407,2.6.57.3060348.2899,1520,09:25:00,Diagonal - Pg de Gràcia,2.6.2899,6,Pg. Manuel Girona
1408,2.6.57.3060348.2899,3932,09:42:00,Pg Manuel Girona - Benet Mateu - Final de traj...,2.6.2899,6,Pg. Manuel Girona
...,...,...,...,...,...,...,...
68802,2.250.102.3185372.3051,1595,11:08:00,Centre Esportiu Ciutat Meridiana - Final trajecte,2.250.3051,D50,Ciutat Meridiana
68803,2.250.102.3185373.3051,1595,11:18:00,Centre Esportiu Ciutat Meridiana - Final trajecte,2.250.3051,D50,Paral·lel
68804,2.250.102.3185373.3051,1622,11:26:00,S'Agaró - Sa Tuna,2.250.3051,D50,Paral·lel
68805,2.250.102.3185373.3051,819,11:39:00,Pg Verdum - Pl de la República - Llucmajor,2.250.3051,D50,Paral·lel


In [671]:
times_w_routes_tmb['route_short_name'].unique()

array(['6', '7', '13', '19', '21', '22', '23', '24', '27', '33', '34',
       '39', '46', '47', '52', '54', '55', '59', '60', '62', '63', '65',
       '67', '68', '70', '76', '78', '91', '94', '95', '96', '97', '102',
       '104', '107', '109', '111', '112', '113', '114', '115', '117',
       '118', '119', '120', '121', '122', '123', '124', '125', '126',
       '127', '128', '129', '130', '131', '132', '133', '134', '136',
       '137', '138', '141', '150', '157', '175', '180', '182', '183',
       '185', '191', '192', '196', 'V1', 'H2', 'V3', 'H4', 'V5', 'H6',
       'V7', 'H8', 'V9', 'H10', 'V11', 'H12', 'V13', 'H14', 'V15', 'H16',
       'V17', 'V19', 'D20', 'V21', 'V23', 'V25', 'V27', 'V29', 'V31',
       'V33', 'D40', 'X1', 'X2', 'X3', 'D50'], dtype=object)

In [672]:
times_w_routes_tmb[times_w_routes_tmb['route_short_name'] == '102'].groupby('trip_id')['stop_id'].nunique().sort_values(ascending=False)

trip_id
2.102.15.3134507.2876    4
2.102.15.3134508.2876    4
2.102.15.3134509.2876    4
Name: stop_id, dtype: int64

In [673]:
times_w_routes_tmb[times_w_routes_tmb['route_short_name'] == '102']['stop_id'].nunique()

6

## Together

In [674]:
times_w_routes = pd.concat([times_w_routes_tmb, times_w_routes_amb], ignore_index=True)
times_w_routes

,trip_id,stop_id,departure_time,stop_name,route_id,route_short_name,trip_headsign
0,2.6.57.3060348.2899,3257,09:00:00,Llacuna - Ramon Turró - Final de trajecte,2.6.2899,6,Pg. Manuel Girona
1,2.6.57.3060348.2899,3400,09:07:00,Marina - Meridiana,2.6.2899,6,Pg. Manuel Girona
2,2.6.57.3060348.2899,317,09:15:00,Pl Tetuan - Pg de Sant Joan,2.6.2899,6,Pg. Manuel Girona
3,2.6.57.3060348.2899,1520,09:25:00,Diagonal - Pg de Gràcia,2.6.2899,6,Pg. Manuel Girona
4,2.6.57.3060348.2899,3932,09:42:00,Pg Manuel Girona - Benet Mateu - Final de traj...,2.6.2899,6,Pg. Manuel Girona
...,...,...,...,...,...,...,...
247638,297.13.1.1.14,112100,1900-01-01 10:55:48,Dr. Robert - Sardana,297,B8,Bonavista
247639,297.13.1.1.15,112100,1900-01-01 11:10:48,Dr. Robert - Sardana,297,B8,Bonavista
247640,297.13.1.1.16,112100,1900-01-01 11:25:48,Dr. Robert - Sardana,297,B8,Bonavista
247641,297.13.1.1.17,112100,1900-01-01 11:40:48,Dr. Robert - Sardana,297,B8,Bonavista


In [675]:
times_w_routes['departure_time'] = pd.to_datetime(times_w_routes['departure_time'], format='%H:%M:%S')
times_w_routes = times_w_routes.sort_values(['stop_id', 'route_id', 'trip_headsign','departure_time'])

times_w_routes['interval_minutes'] = (
    times_w_routes.groupby(['stop_id', 'route_id','trip_headsign'])['departure_time']
    .diff()
    .dt.total_seconds()
    .div(60)
 )

times_w_routes['interval_minutes'] = times_w_routes['interval_minutes'].apply(lambda x: np.nan if x < 0 else x)
avg_int_len = (
    times_w_routes.dropna(subset=['interval_minutes'])
    .groupby(['route_id', 'route_short_name', 'stop_id', 'stop_name'], as_index=False)['interval_minutes']
    .mean()
).rename(columns={'interval_minutes': 'avg_interval_minutes'})
int_var = times_w_routes.groupby([ 'route_id', 'route_short_name', 'stop_id', 'stop_name'], as_index=False)['interval_minutes'].var().rename(columns={'interval_minutes': 'var_interval_minutes'})
avg_wait_times = avg_int_len.merge(int_var, on=['route_id', 'route_short_name', 'stop_id', 'stop_name'], how='left')
avg_wait_times['wait_time'] =((avg_wait_times['avg_interval_minutes'] ** 2) + avg_wait_times['var_interval_minutes']) /( 2*avg_wait_times['avg_interval_minutes'])




avg_wait_times = avg_wait_times.groupby(['route_id','route_short_name', 'stop_id', 'stop_name'], as_index=False)['wait_time'].mean()
avg_wait_times['stop_id'] = 'B' + '-' + avg_wait_times['route_short_name'] + '-' + avg_wait_times['stop_id']
avg_wait_times

,route_id,route_short_name,stop_id,stop_name,wait_time
0,129,87,B-87-101341,Baixada de la Plana - Tajo,2.442666
1,129,87,B-87-102722,Llobregós - Murtra,2.350146
2,129,87,B-87-102723,Llobregós - Pantà de Tremp,2.350146
3,129,87,B-87-104298,Pg. Mare de Déu del Coll - Santuari,2.439896
4,129,87,B-87-106167,Llobregós - Fastenrath,2.359420
...,...,...,...,...,...
6487,2.96.2873,96,B-96-3776,Bach - Joan Miró - Final de trajecte,6.221545
6488,2.96.2873,96,B-96-3952,"av. Ribera, 13-54",6.919852
6489,2.96.2873,96,B-96-9968,"Juan de Garay, 116",7.415896
6490,2.97.2874,97,B-97-2718,Pl Primer de Maig - Final de trajecte,6.245185


In [676]:
exchange_edges_bus = exchange_edges_bus.merge(avg_wait_times[['stop_id','wait_time']], left_on='dest', right_on='stop_id', how='left')
exchange_edges_bus = exchange_edges_bus[['origen', 'dest', 'tram', 'mode','lines','type','time',  'wait_time','directed','geometry']]
exchange_edges_bus[exchange_edges_bus['wait_time'].isna()]

,origen,dest,tram,mode,lines,type,time,wait_time,directed,geometry
0,SB-2,B-59-2,Av Icària - Àlaba - Av Icària - Àlaba,Bus - Bus,IU Stop - 59,Exchange - Self,1.00,NaN,True,POINT (2.198984998801282 41.393104003754864)
1,B-H16-2,B-59-2,Av Icària - Àlaba - Av Icària - Àlaba,Bus - Bus,H16 - 59,Exchange - Self,1.00,NaN,True,POINT (2.198984998801282 41.393104003754864)
2,B-V27-2,B-59-2,Av Icària - Àlaba - Av Icària - Àlaba,Bus - Bus,V27 - 59,Exchange - Self,1.00,NaN,True,POINT (2.198984998801282 41.393104003754864)
3,SB-2,B-H16-2,Av Icària - Àlaba - Av Icària - Àlaba,Bus - Bus,IU Stop - H16,Exchange - Self,1.00,NaN,True,POINT (2.198984998801282 41.393104003754864)
4,B-59-2,B-H16-2,Av Icària - Àlaba - Av Icària - Àlaba,Bus - Bus,59 - H16,Exchange - Self,1.00,NaN,True,POINT (2.198984998801282 41.393104003754864)
...,...,...,...,...,...,...,...,...,...,...
147066,B-86-2023,B-86-112354,Calderón de la Barca - Ctra del Carmel - Ctra....,Bus - Bus,86 - 86,Exchange,4.45,NaN,True,"LINESTRING (2.157137 41.4200565, 2.1572211 41...."
147067,B-24-9425,B-86-112354,"Carmel, 118 - Ctra. del Carmel - Albert Llanas",Bus - Bus,24 - 86,Exchange,3.20,NaN,True,"LINESTRING (2.1574434 41.4180324, 2.1573509 41..."
147068,B-86-9425,B-86-112354,"Carmel, 118 - Ctra. del Carmel - Albert Llanas",Bus - Bus,86 - 86,Exchange,3.20,NaN,True,"LINESTRING (2.1574434 41.4180324, 2.1573509 41..."
147069,B-V19-9425,B-86-112354,"Carmel, 118 - Ctra. del Carmel - Albert Llanas",Bus - Bus,V19 - 86,Exchange,3.20,NaN,True,"LINESTRING (2.1574434 41.4180324, 2.1573509 41..."


In [677]:
na_buses = exchange_edges_bus[exchange_edges_bus['wait_time'].isna()]
na_buses['line'] = exchange_edges_bus['lines'].str.split(' - ').str[1]
na_buses['dest_stop'] = exchange_edges_bus['dest'].str.split('-').str[2]
exchange_edges_bus = exchange_edges_bus[exchange_edges_bus['wait_time'].notna()]
na_buses['line'] = na_buses['lines'].str.split(' - ').str[1]
na_buses['line'].unique()

/var/folders/95/s4thp5290fd41pw2cj8713k80000gn/T/ipykernel_51408/2304860780.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  na_buses['line'] = exchange_edges_bus['lines'].str.split(' - ').str[1]
/var/folders/95/s4thp5290fd41pw2cj8713k80000gn/T/ipykernel_51408/2304860780.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  na_buses['dest_stop'] = exchange_edges_bus['dest'].str.split('-').str[2]


array(['59', 'H16', 'V27', 'H10', '13', '150', 'V33', '60', 'H8', '133',
       '27', 'D40', '7', '33', '67', '113', 'H6', '175', '22', 'V17',
       '47', 'V21', '136', 'H12', 'V25', '6', '19', '54', 'V19', '63',
       '78', '34', '109', 'V3', '76', 'H4', 'V11', '24', 'V15', 'V13',
       'V9', '39', 'V23', 'D50', '124', 'H2', '196', '65', '52', '123',
       '68', 'V7', '96', 'V31', '55', '91', '120', 'D20', '114', '70',
       '119', 'V5', '141', '116', '130', '23', '157', '117', '185', '192',
       '191', 'H14', '62', '104', '121', 'V1', 'V29', '182', '125', '102',
       '86', '126', '122', '129', '131', 'B24', '21', '127', '112', 'B25',
       '180', '115', '97', '183', '132', 'X1', '128', '118', '199', 'B14',
       'B21', 'B23', 'B20', '107', '111'], dtype=object)

In [678]:
na_buses

,origen,dest,tram,mode,lines,type,time,wait_time,directed,geometry,line,dest_stop
0,SB-2,B-59-2,Av Icària - Àlaba - Av Icària - Àlaba,Bus - Bus,IU Stop - 59,Exchange - Self,1.00,NaN,True,POINT (2.198984998801282 41.393104003754864),59,2
1,B-H16-2,B-59-2,Av Icària - Àlaba - Av Icària - Àlaba,Bus - Bus,H16 - 59,Exchange - Self,1.00,NaN,True,POINT (2.198984998801282 41.393104003754864),59,2
2,B-V27-2,B-59-2,Av Icària - Àlaba - Av Icària - Àlaba,Bus - Bus,V27 - 59,Exchange - Self,1.00,NaN,True,POINT (2.198984998801282 41.393104003754864),59,2
3,SB-2,B-H16-2,Av Icària - Àlaba - Av Icària - Àlaba,Bus - Bus,IU Stop - H16,Exchange - Self,1.00,NaN,True,POINT (2.198984998801282 41.393104003754864),H16,2
4,B-59-2,B-H16-2,Av Icària - Àlaba - Av Icària - Àlaba,Bus - Bus,59 - H16,Exchange - Self,1.00,NaN,True,POINT (2.198984998801282 41.393104003754864),H16,2
...,...,...,...,...,...,...,...,...,...,...,...,...
147066,B-86-2023,B-86-112354,Calderón de la Barca - Ctra del Carmel - Ctra....,Bus - Bus,86 - 86,Exchange,4.45,NaN,True,"LINESTRING (2.157137 41.4200565, 2.1572211 41....",86,112354
147067,B-24-9425,B-86-112354,"Carmel, 118 - Ctra. del Carmel - Albert Llanas",Bus - Bus,24 - 86,Exchange,3.20,NaN,True,"LINESTRING (2.1574434 41.4180324, 2.1573509 41...",86,112354
147068,B-86-9425,B-86-112354,"Carmel, 118 - Ctra. del Carmel - Albert Llanas",Bus - Bus,86 - 86,Exchange,3.20,NaN,True,"LINESTRING (2.1574434 41.4180324, 2.1573509 41...",86,112354
147069,B-V19-9425,B-86-112354,"Carmel, 118 - Ctra. del Carmel - Albert Llanas",Bus - Bus,V19 - 86,Exchange,3.20,NaN,True,"LINESTRING (2.1574434 41.4180324, 2.1573509 41...",86,112354


In [679]:
avg_wait_times_line

,route_short_name,avg_wait_time
0,102,NaN
1,104,35.000000
2,107,19.313178
3,109,4.593736
4,111,12.120226
...,...,...
212,X83,7.470716
213,X84,8.666867
214,X86,5.776288
215,X95,6.897963


In [680]:
avg_wait_times_line = avg_wait_times.groupby(['route_short_name'], as_index=False)['wait_time'].mean().rename(columns={'wait_time': 'avg_wait_time'})
na_buses = na_buses.merge(avg_wait_times_line, left_on='line', right_on='route_short_name', how='left')
na_buses['wait_time'] = na_buses['avg_wait_time']
no_longer_na_buses  = na_buses[na_buses['wait_time'].notna()]
no_longer_na_buses.drop(columns=['line','dest_stop','avg_wait_time','route_short_name'], inplace=True)
exchange_edges_bus = pd.concat([exchange_edges_bus, no_longer_na_buses], ignore_index=True)
na_buses = na_buses[na_buses['wait_time'].isna()]
na_buses

/var/folders/95/s4thp5290fd41pw2cj8713k80000gn/T/ipykernel_51408/2629767473.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  no_longer_na_buses.drop(columns=['line','dest_stop','avg_wait_time','route_short_name'], inplace=True)


,origen,dest,tram,mode,lines,type,time,wait_time,directed,geometry,line,dest_stop,route_short_name,avg_wait_time
748,SB-187,B-116-187,Escorial - Clínica Nostra Senyora del Remei - ...,Bus - Bus,IU Stop - 116,Exchange - Self,1.00,NaN,True,POINT (2.1590310009630698 41.40980399586584),116,187,NaN,NaN
749,B-39-187,B-116-187,Escorial - Clínica Nostra Senyora del Remei - ...,Bus - Bus,39 - 116,Exchange - Self,1.00,NaN,True,POINT (2.1590310009630698 41.40980399586584),116,187,NaN,NaN
1426,SB-365,B-102-365,Lletres - Dante Alighieri - Lletres - Dante Al...,Bus - Bus,IU Stop - 102,Exchange - Self,1.00,NaN,True,POINT (2.157915000707638 41.42793599871785),102,365,102,NaN
1427,B-87-365,B-102-365,Lletres - Dante Alighieri - Lletres - Dante Al...,Bus - Bus,87 - 102,Exchange - Self,1.00,NaN,True,POINT (2.157915000707638 41.42793599871785),102,365,102,NaN
1428,B-V21-365,B-102-365,Lletres - Dante Alighieri - Lletres - Dante Al...,Bus - Bus,V21 - 102,Exchange - Self,1.00,NaN,True,POINT (2.157915000707638 41.42793599871785),102,365,102,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
98731,B-86-2023,B-86-112354,Calderón de la Barca - Ctra del Carmel - Ctra....,Bus - Bus,86 - 86,Exchange,4.45,NaN,True,"LINESTRING (2.157137 41.4200565, 2.1572211 41....",86,112354,NaN,NaN
98732,B-24-9425,B-86-112354,"Carmel, 118 - Ctra. del Carmel - Albert Llanas",Bus - Bus,24 - 86,Exchange,3.20,NaN,True,"LINESTRING (2.1574434 41.4180324, 2.1573509 41...",86,112354,NaN,NaN
98733,B-86-9425,B-86-112354,"Carmel, 118 - Ctra. del Carmel - Albert Llanas",Bus - Bus,86 - 86,Exchange,3.20,NaN,True,"LINESTRING (2.1574434 41.4180324, 2.1573509 41...",86,112354,NaN,NaN
98734,B-V19-9425,B-86-112354,"Carmel, 118 - Ctra. del Carmel - Albert Llanas",Bus - Bus,V19 - 86,Exchange,3.20,NaN,True,"LINESTRING (2.1574434 41.4180324, 2.1573509 41...",86,112354,NaN,NaN


In [681]:
na_buses['line'].unique()

array(['116', '102', '86', '199'], dtype=object)

In [ ]:
exchange_edges_bus

In [ ]:
avg_wait_times_line

# Metro

In [ ]:
exchange_edges_metro = exchange_edges[exchange_edges['dest'].str.startswith('M')]
exchange_edges_metro['lines'].value_counts()

In [ ]:
stop_times = pd.read_table('Data/GTFS_TMB/stop_times.txt', sep=',')[['trip_id', 'stop_id', 'departure_time']]
stop_times = stop_times[stop_times['departure_time'].notna()]
stop_times = stop_times[stop_times['departure_time'].str.slice(0,2).astype(int) >= 7]
stop_times = stop_times[stop_times['departure_time'].str.slice(0,2).astype(int) <= 11]
stop_times

In [ ]:
stops = pd.read_table('Data/GTFS_TMB/stops.txt', sep=',')[['stop_id','stop_name']]
stops

In [ ]:
stops_w_times = stops.merge(stop_times, on='stop_id', how='left')
stops_w_times = stops_w_times[stops_w_times['trip_id'].notna()]
stops_w_times

In [ ]:
trips = pd.read_table('Data/GTFS_TMB/trips.txt', sep=',')[['route_id','trip_id','trip_headsign']]
routes = pd.read_table('Data/GTFS_TMB/routes.txt', sep=',')[['route_id','route_short_name']]
metro = ['L1','L2','L3','L4','L5']
trips = routes.merge(trips, on='route_id', how='left')
trips = trips[trips['route_short_name'].isin(metro)]
trips

In [ ]:
times_w_routes = stops_w_times.merge(trips, on='trip_id', how='left')
times_w_routes.drop_duplicates(subset=['stop_id','stop_name','departure_time','route_id','route_short_name','trip_headsign'], keep='first', inplace=True)
times_w_routes = times_w_routes[times_w_routes['route_id'].notna()]
times_w_routes

In [ ]:
times_w_routes[times_w_routes['route_short_name'] == 'L1'].groupby('trip_id')['stop_id'].nunique()

In [ ]:
times_w_routes[times_w_routes['route_short_name'] == 'L1']['stop_id'].nunique()

In [ ]:
times_w_routes['departure_time'] = pd.to_datetime(times_w_routes['departure_time'], format='%H:%M:%S')
times_w_routes = times_w_routes.sort_values(['stop_id', 'route_id','route_short_name','trip_headsign', 'departure_time'])

times_w_routes['interval_minutes'] = (
    times_w_routes.groupby(['stop_id', 'route_id'])['departure_time']
    .diff()
    .dt.total_seconds()
    .div(60)
 )

times_w_routes['interval_minutes'] = times_w_routes['interval_minutes'].apply(lambda x: np.nan if x < 0 else x)
avg_wait_times = (
    times_w_routes.dropna(subset=['interval_minutes'])
    .assign(wait_time=lambda df: df['interval_minutes'] / 2)
    .groupby(['route_id','route_short_name', 'stop_id', 'stop_name','trip_headsign'], as_index=False)['wait_time']
    .mean()
)
avg_wait_times = avg_wait_times.groupby(['route_id','route_short_name', 'stop_id', 'stop_name'], as_index=False)['wait_time'].mean()
avg_wait_times

In [ ]:
exchange_edges_metro['dest_name'] = exchange_edges_metro['tram'].str.split(' - ').str[-1]
exchange_edges_metro['dest_name'] = exchange_edges_metro['dest_name'].replace({'Av. de Xile':'Ernest Lluch'})

In [ ]:
exchange_edges_metro = exchange_edges_metro.merge(avg_wait_times[['stop_name','wait_time']], left_on='dest_name', right_on='stop_name', how='left')
exchange_edges_metro = exchange_edges_metro[['origen', 'dest','dest_name','tram', 'mode','lines','type','time',  'wait_time','directed','geometry']]
exchange_edges_metro

In [ ]:
exchange_edges_metro[exchange_edges_metro['wait_time'].isna()]

# Save

In [ ]:
exchanges = pd.concat([exchange_edges_tram, exchange_edges_fgc, exchange_edges_bus, exchange_edges_metro], ignore_index=True)
exchanges.to_csv('Edges/Exchanges_with_Wait_Times.csv', index=False)